# NEI2022v2 + CAMS v6.2 Diagnostic Checks

Clean diagnostic notebook for the 2023 NEI2022v2-merged CAMS workflow.
This version removes legacy NEI2017 content and loads all runtime paths from `config/paths.json`.


## 1) Imports


In [ ]:
from pathlib import Path
import re
import sys

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import cm


## 2) Load Config (Single Source of Paths)


In [ ]:
# Resolve repo root from this notebook location
REPO = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if not (REPO / 'src').exists():
    raise FileNotFoundError(f'Cannot find repo root from cwd={Path.cwd()}')

sys.path.insert(0, str(REPO / 'src'))
from nei_merge.config import load_json_config

CONFIG_PATH = REPO / 'config' / 'paths.json'
cfg = load_json_config(CONFIG_PATH)
paths = cfg['paths']
workflow = cfg['workflow']

DATE_TAG = workflow['date_tag']
OUTPUT_YEAR = workflow['output_year']

MAPPED_DIR = Path(paths['mapped_species_dir'])
CAMS_NE0_DIR = Path(paths['cams_ne0conus_monthly_dir'])
MASK_FILE = Path(paths['conus_mask_80km_file'])

# Optional post-processed dirs
CONUS_ZERO_DIR = Path(paths.get('conus_zerooutside_temp_dir', '')) if paths.get('conus_zerooutside_temp_dir') else None
CONUS_FIXED_DIR = Path(paths.get('conus_zerooutside_fixed_dir', '')) if paths.get('conus_zerooutside_fixed_dir') else None

print('Config loaded:', CONFIG_PATH)
print('MAPPED_DIR:', MAPPED_DIR)
print('CAMS_NE0_DIR:', CAMS_NE0_DIR)
print('MASK_FILE:', MASK_FILE)


## 3) Helper Functions


In [ ]:
rx_nei = re.compile(rf"Y{OUTPUT_YEAR}_CAMS-GLOB-ANTv6\.2_conusNEI2022v2_ne0CONUSne30x8_(?P<spc>.+?)_{DATE_TAG}\.nc$")
rx_cams = re.compile(r"CAMS-GLOB-ANT_ne0conus30x8_(?P<spc>.+?)_v6\.2_monthly\.nc$")

def species_from_files(folder: Path, regex: re.Pattern):
    out = []
    for f in sorted(folder.glob('*.nc')):
        m = regex.search(f.name)
        if m:
            out.append(m.group('spc'))
    return sorted(set(out))

def mapped_file(species: str) -> Path:
    return MAPPED_DIR / f"Y{OUTPUT_YEAR}_CAMS-GLOB-ANTv6.2_conusNEI2022v2_ne0CONUSne30x8_{species}_{DATE_TAG}.nc"

def cams_file(species: str) -> Path:
    return CAMS_NE0_DIR / f"CAMS-GLOB-ANT_ne0conus30x8_{species}_v6.2_monthly.nc"


## 4) Species Intersection (Mapped vs CAMS Reference)


In [ ]:
mapped_species = species_from_files(MAPPED_DIR, rx_nei)
cams_species = species_from_files(CAMS_NE0_DIR, rx_cams)
common_species = sorted(set(mapped_species).intersection(cams_species))
only_mapped = sorted(set(mapped_species) - set(cams_species))
only_cams = sorted(set(cams_species) - set(mapped_species))

print('Mapped count:', len(mapped_species))
print('CAMS count  :', len(cams_species))
print('Common count:', len(common_species))
print('Only mapped :', only_mapped)
print('Only CAMS   :', only_cams)

common_species[:20]


## 5) Load One Species and Standardize Variable Name


In [ ]:
species = 'NO'  # change as needed

f_nei = mapped_file(species)
f_cams = cams_file(species)
print(f_nei)
print(f_cams)

ds_nei = xr.open_dataset(f_nei)
ds_cams = xr.open_dataset(f_cams)

# Some files use 'sum', some use 'emiss'
var_nei = 'emiss' if 'emiss' in ds_nei.data_vars else 'sum'
var_cams = 'emiss' if 'emiss' in ds_cams.data_vars else 'sum'

nei_da = ds_nei[var_nei]
cams_da = ds_cams[var_cams]

print('NEI var:', var_nei, 'dims=', nei_da.dims, 'shape=', nei_da.shape)
print('CAMS var:', var_cams, 'dims=', cams_da.dims, 'shape=', cams_da.shape)
print('Units:', nei_da.attrs.get('units'), '|', cams_da.attrs.get('units'))


## 6) Monthly Mean Difference (Flexible Month Length)


In [ ]:
Monthi = '2023-07'  # YYYY-MM

mask_nei = nei_da.time.dt.strftime('%Y-%m') == Monthi
mask_cams = cams_da.time.dt.strftime('%Y-%m') == Monthi

nei_mean = nei_da.sel(time=mask_nei).mean(dim='time')
cams_mean = cams_da.sel(time=mask_cams).mean(dim='time')
diff = nei_mean - cams_mean

mask_da = xr.open_dataarray(MASK_FILE)
diff_conus = xr.where(mask_da, diff, 0.0)

print('NEI ntime in month :', int(mask_nei.sum()))
print('CAMS ntime in month:', int(mask_cams.sum()))
print('Inside-mask min/max:', float(diff_conus.where(mask_da).min()), float(diff_conus.where(mask_da).max()))
print('Inside-mask mean abs:', float(np.abs(diff_conus.where(mask_da)).mean()))


## 7) Plot Difference on ncol Index (Fast Visual QC)


In [ ]:
Emis_Scalefactor = 1e11
setmap = cm.RdBu_r
rangeMax = 0.08  # fixed range for cross-month comparability

plot_ar = (diff_conus.values / Emis_Scalefactor).astype(float)
eps = 0.02 * rangeMax
plot_ar = np.where(np.abs(plot_ar) < eps, 0.0, plot_ar)

fig = plt.figure(figsize=(12, 3.8))
ax = fig.add_subplot(1, 1, 1)
im = ax.scatter(np.arange(plot_ar.size), plot_ar, c=plot_ar, s=0.4, cmap=setmap, vmin=-rangeMax, vmax=rangeMax)
ax.set_title(f'{species} {Monthi} (NEI monthly mean - CAMS monthly mean)')
ax.set_xlabel('ncol index')
ax.set_ylabel(f"{nei_da.attrs.get('units', 'value')} / {Emis_Scalefactor:.0e}")
plt.colorbar(im, ax=ax, label='Difference (scaled)')
plt.tight_layout()
plt.show()


## 8) Optional: Print Species-to-Path Lines for Run Lists


In [ ]:
for spc in common_species:
    print(f"'{spc}  -> {mapped_file(spc)}',")


## 9) Close Datasets


In [ ]:
ds_nei.close()
ds_cams.close()
